In [ ]:
%pip install openai
%pip install git+https://github.com/supabase-community/supabase-py

In [17]:
import os
import psycopg2
from psycopg2.extras import RealDictCursor

db_host = os.environ.get("SUPABASE_DB_HOST")
db_name_env = os.environ.get("SUPABASE_DB_NAME")
db_user = os.environ.get("SUPABASE_DB_USER")
db_password = os.environ.get("SUPABASE_DB_PASSWORD")
db_port = os.environ.get("SUPABASE_DB_PORT", "5432")

if not all([db_host, db_name_env, db_user, db_password]):
    raise ValueError("Database connection parameters must be set in environment variables")

def get_db_connection():
    return  psycopg2.connect(
                host=db_host,
                database=db_name_env,  # Use the environment variable value
                user=db_user,
                password=db_password,
                port=db_port,
                cursor_factory=RealDictCursor  # This makes cursor return dictionaries
            )

In [14]:
# create indexes

conn = get_db_connection()
conn.autocommit = True
cursor = conn.cursor()

cursor.execute("""
    CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_topics_topic_solicitation
    ON db1.topics(topic_number, solicitation_id);
""")

In [ ]:
### EMBED TOPICS ###

from openai import OpenAI
from supabase import create_client, Client
from server.app.services.db import get_db_connection, get_db_cursor
import time
import dotenv

dotenv.load_dotenv("./server/.env")

OpenAI.api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI()
BATCH_SIZE = 300

# === Embedding Function ===
def get_embeddings_batch(texts: list[str]) -> list[list[float]]:
    try:
        response = client.embeddings.create(
            input=texts,
            model="text-embedding-3-small"
        )
        return [item.embedding for item in response.data]
    except Exception as e:
        print("Embedding error:", e)
        return [None] * len(texts)

# === Combine Fields ===
def combine_fields(row, solicitation): 
    #remove topic dates, name, sbir/sttr, phase. just include info relevant to llm
    return f"""
    Topic Title: {row.get('topic_title', '')}
    Topic Description: {row.get('topic_description', '')}
    Branch: {row.get('branch', '')}
    Topic Open Date: {row.get('topic_open_date', '')}
    Topic Closed Date: {row.get('topic_closed_date', '')}
    Topic POC Name: {row.get('tpoc_name', '')}
    
    Parent Solicitation Title: {solicitation.get('solicitation_title', '')}
    SBIR or STTR Program: {solicitation.get('program', '')}
    Phase I or Phase II: {solicitation.get('phase', '')}
    Solicitation Agency: {solicitation.get('agency', '')}
    Solicitation Branch: {solicitation.get('branch', '')}
    Solicitation Year: {solicitation.get('solicitation_year', '')}
    """

# === Load and Update ===
def embed_and_update_rows(db_name):
    try:
        conn = psycopg2.connect(
            host=db_host,
            database=db_name_env,  # Use the environment variable value
            user=db_user,
            password=db_password,
            port=db_port,
            cursor_factory=RealDictCursor  # This makes cursor return dictionaries
        )
    
        cursor = conn.cursor()
        cursor.execute("SET statement_timeout = '60s';")

        # Fetch rows with NULL embeddings
        cursor.execute("SELECT * FROM db1.topics WHERE embedding IS NULL LIMIT 2000")
        rows = cursor.fetchall()

        if not rows:
            print("No rows to process.")
            return
        
        print(f"Processing {len(rows)} rows...")

        # Group into batches
        for i in range(0, len(rows), BATCH_SIZE):
            batch = rows[i:i + BATCH_SIZE]
            texts = []
            keys = []

            # Build input batch
            for row in batch:
                cursor.execute("SELECT * FROM db1.solicitations WHERE solicitation_id = %s", (row['solicitation_id'],))
                solicitation = cursor.fetchone()
                if solicitation:
                    combined = combine_fields(row, solicitation)
                    texts.append(combined)
                    keys.append((row['topic_number'], row['solicitation_id']))

            # Embed the batch
            embeddings = get_embeddings_batch(texts)

            # Bulk update
            for idx, embedding in enumerate(embeddings):
                if embedding:
                    cursor.execute(
                        "UPDATE db1.topics SET embedding = %s WHERE topic_number = %s AND solicitation_id = %s::bigint",
                        (embedding, keys[idx][0], keys[idx][1])
                    )
            conn.commit()
            print(f"✅ Updated {len(embeddings)} rows (batch {i // BATCH_SIZE + 1})")


    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()
    
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

embed_and_update_rows("db1")

In [ ]:
# === EMBED AWARDS ===

from openai import OpenAI
from supabase import create_client, Client
from server.app.services.db import get_db_connection, get_db_cursor
import time
import dotenv

dotenv.load_dotenv("./server/.env")

OpenAI.api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI()
BATCH_SIZE = 1000

# === Embedding Function ===
def get_embeddings_batch(texts: list[str]) -> list[list[float]]:
    try:
        response = client.embeddings.create(
            input=texts,
            model="text-embedding-3-small"
        )
        return [item.embedding for item in response.data]
    except Exception as e:
        print("Embedding error:", e)
        return [None] * len(texts)

# === Combine Fields ===
def combine_award_fields(row): 
    return f"""
    Company: {row.get('firm', '')}
    Award Title: {row.get('award', '')}
    Abstract: {row.get('abstract', '')}
    Agency: {row.get('agency', '')}
    Branch: {row.get('branch', '')}
    Phase: {row.get('phase', '')}
    Program: {row.get('program', '')}
    Award Amount: {row.get('award_amount', '')}
    Address: {row.get('address1', '')} {row.get('address2', '')} {row.get('city', '')} {row.get('state', '')} {row.get('zip', '')}
    """

# === Load and Update ===
def embed_and_update_rows(db_name):
    try:
        conn = psycopg2.connect(
            host=db_host,
            database=db_name_env,  # Use the environment variable value
            user=db_user,
            password=db_password,
            port=db_port,
            cursor_factory=RealDictCursor  # This makes cursor return dictionaries
        )
    
        cursor = conn.cursor()
        cursor.execute("SET statement_timeout = '60s';")

        # Fetch rows with NULL embeddings
        cursor.execute("SELECT * FROM db2.awards WHERE embedding IS NULL LIMIT 10000")
        rows = cursor.fetchall()

        if not rows:
            print("No rows to process.")
            return
        
        print(f"Processing {len(rows)} rows...")

        # Group into batches
        for i in range(0, len(rows), BATCH_SIZE):
            batch = rows[i:i + BATCH_SIZE]
            texts = []
            keys = []

            # Build input batch
            for row in batch:
                combined = combine_award_fields(row)
                texts.append(combined)
                keys.append(row['award_link'])

            # Embed the batch
            embeddings = get_embeddings_batch(texts)

            # Bulk update
            for idx, embedding in enumerate(embeddings):
                if embedding:
                    cursor.execute(
                        "UPDATE db2.awards SET embedding = %s WHERE award_link = %s::bigint",
                        (embedding, keys[idx])
                    )
            conn.commit()
            print(f"✅ Updated {len(embeddings)} rows (batch {i // BATCH_SIZE + 1})")


    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()
    
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

embed_and_update_rows("db1")

In [ ]:
# === EMBED AWARDS - ASYNC [DRAFT] ===

from openai import OpenAI
from supabase import create_client, Client
from server.app.services.db import get_db_connection, get_db_cursor
import time
import dotenv
import asyncio
import httpx
from tqdm import tqdm

dotenv.load_dotenv("./server/.env")

OpenAI.api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI()

BATCH_SIZE = 600
PARALLEL_WORKERS = 5

# === Fetch next batch from DB ===
def fetch_next_batch(cursor, limit):
    cursor.execute("""
        SELECT * FROM db2.awards
        WHERE embedding IS NULL
        LIMIT %s
    """, (limit,))
    return cursor.fetchall()

# === Embedding Function ===
def get_embeddings_batch(texts: list[str]) -> list[list[float]]:
    try:
        response = client.embeddings.create(
            input=texts,
            model="text-embedding-3-small"
        )
        return [item.embedding for item in response.data]
    except Exception as e:
        print("Embedding error:", e)
        return [None] * len(texts)

# === Combine Fields ===
def combine_award_fields(row): 
    return f"""
    Company: {row.get('firm', '')}
    Award Title: {row.get('award', '')}
    Abstract: {row.get('abstract', '')}
    Agency: {row.get('agency', '')}
    Branch: {row.get('branch', '')}
    Phase: {row.get('phase', '')}
    Program: {row.get('program', '')}
    Award Amount: {row.get('award_amount', '')}
    Address: {row.get('address1', '')} {row.get('address2', '')} {row.get('city', '')} {row.get('state', '')} {row.get('zip', '')}
    """

# === Load and Update ===
def embed_and_update_rows(db_name):
    try:
        conn = psycopg2.connect(
            host=db_host,
            database=db_name_env,  # Use the environment variable value
            user=db_user,
            password=db_password,
            port=db_port,
            cursor_factory=RealDictCursor  # This makes cursor return dictionaries
        )
    
        cursor = conn.cursor()
        cursor.execute("SET statement_timeout = '60s';")

        # Fetch rows with NULL embeddings
        cursor.execute("SELECT * FROM db2.awards WHERE embedding IS NULL LIMIT 6000")
        rows = cursor.fetchall()

        if not rows:
            print("No rows to process.")
            return
        
        print(f"Processing {len(rows)} rows...")

        # Group into batches
        for i in range(0, len(rows), BATCH_SIZE):
            batch = rows[i:i + BATCH_SIZE]
            texts = []
            keys = []

            # Build input batch
            for row in batch:
                combined = combine_award_fields(row)
                texts.append(combined)
                keys.append(row['award_link'])

            # Embed the batch
            embeddings = get_embeddings_batch(texts)

            # Bulk update
            for idx, embedding in enumerate(embeddings):
                if embedding:
                    cursor.execute(
                        "UPDATE db2.awards SET embedding = %s WHERE award_link = %s::bigint",
                        (embedding, keys[idx])
                    )
            conn.commit()
            print(f"✅ Updated {len(embeddings)} rows (batch {i // BATCH_SIZE + 1})")


    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()
    
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

embed_and_update_rows("db1")

In [ ]:
# === EMBED COMPANIES === have not fixed yet

from openai import OpenAI
from supabase import create_client, Client
from server.app.services.db import get_db_connection, get_db_cursor
import time
import dotenv

dotenv.load_dotenv("./server/.env")

OpenAI.api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI()
BATCH_SIZE = 300

# === Embedding Function ===
def get_embeddings_batch(texts: list[str]) -> list[list[float]]:
    try:
        response = client.embeddings.create(
            input=texts,
            model="text-embedding-3-small"
        )
        return [item.embedding for item in response.data]
    except Exception as e:
        print("Embedding error:", e)
        return [None] * len(texts)

# === Combine Fields ===
def combine_company_fields(row): 
    return f"""
    Company: {row.get('company_name', '')}
    UEI: {row.get('uei', '')}
    DUNS: {row.get('duns', '')}
    Address: {row.get('address1', '')} {row.get('address2', '')} {row.get('city', '')} {row.get('state', '')} {row.get('zip', '')}
    Company URL: {row.get('company_url', '')}
    Woman Owned: {row.get('woman_owned', '')}
    Socially Economically Disadvantaged: {row.get('socially_economically_disadvantaged', '')}
    Hubzone Owned: {row.get('hubzone_owned', '')}h
    Number of Awards: {row.get('number_awards', '')}
    """

# === Load and Update ===
def embed_and_update_rows(db_name):
    try:
        conn = psycopg2.connect(
            host=db_host,
            database=db_name_env,  # Use the environment variable value
            user=db_user,
            password=db_password,
            port=db_port,
            cursor_factory=RealDictCursor  # This makes cursor return dictionaries
        )
    
        cursor = conn.cursor()
        cursor.execute("SET statement_timeout = '60s';")

        # Fetch rows with NULL embeddings
        cursor.execute("SELECT * FROM db3.companies WHERE embedding IS NULL LIMIT 2000")
        rows = cursor.fetchall()

        if not rows:
            print("No rows to process.")
            return
        
        print(f"Processing {len(rows)} rows...")

        # Group into batches
        for i in range(0, len(rows), BATCH_SIZE):
            batch = rows[i:i + BATCH_SIZE]
            texts = []
            keys = []

            # Build input batch
            for row in batch:
                #cursor.execute("SELECT * FROM db1.solicitations WHERE solicitation_id = %s", (row['solicitation_id'],))
                solicitation = cursor.fetchone()
                if solicitation:
                    combined = combine_company_fields(row)
                    texts.append(combined)
                    keys.append((row['topic_number'], row['solicitation_id']))

            # Embed the batch
            embeddings = get_embeddings_batch(texts)

            # Bulk update
            for idx, embedding in enumerate(embeddings):
                if embedding:
                    cursor.execute(
                        "UPDATE db1.topics SET embedding = %s WHERE topic_number = %s AND solicitation_id = %s::bigint",
                        (embedding, keys[idx][0], keys[idx][1])
                    )
            conn.commit()
            print(f"✅ Updated {len(embeddings)} rows (batch {i // BATCH_SIZE + 1})")


    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()
    
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

embed_and_update_rows("db3")